In [44]:
from __future__ import annotations

import torch
from torch import nn

FEATURE_DIM = 2048


class AttentionPool(nn.Module):
    """Learned single-query attention pooling over a variable-length token set.

    Collapses (batch, num_tokens, dim) -> (batch, dim) with a softmax-weighted
    sum instead of a flat average, so the probe can learn to weight the tokens
    that actually matter (e.g. the occluded region of an image) rather than
    diluting them across every patch.
    """

    def __init__(self, dim: int = FEATURE_DIM, key_dim: int = 128):
        super().__init__()
        self.key_proj = nn.Linear(dim, key_dim)
        print(self.key_proj)
        self.query = nn.Parameter(torch.randn(key_dim) * key_dim**-0.5) 
        print(self.query.shape)

    def forward(self, tokens: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        """tokens: (batch, num_tokens, dim). mask: (batch, num_tokens) bool, True = valid."""
        keys = self.key_proj(tokens)  # (b, t, key_dim)
        print(f"keys: {keys.shape}")
        scores = keys @ self.query  # (b, t)
        print(f"scores: {scores.shape}")
        if mask is not None:
            print(f"mask: {mask}")
            scores = scores.masked_fill(~mask, float("-inf"))
        weights = torch.softmax(scores, dim=-1)  # (b, t) # We learn the weights that will serve for the weighted combination.
        print(f"softmax weights: {weights.shape}")
        return torch.einsum("bt,btd->bd", weights, tokens) 


In [45]:
attentionpool = AttentionPool(dim = 2048, key_dim = 128)

Linear(in_features=2048, out_features=128, bias=True)
torch.Size([128])


In [46]:
tokens = torch.randint(low= 1, high= 100, size=(256, FEATURE_DIM), dtype=torch.float32).unsqueeze(0)
print(tokens.shape)
print(tokens.dtype)

torch.Size([1, 256, 2048])
torch.float32


In [47]:
pooled_tokens = attentionpool.forward(tokens=tokens)

keys: torch.Size([1, 256, 128])
scores: torch.Size([1, 256])
softmax weights: torch.Size([1, 256])
